# VECTRI Model: Theory, Equations, and Python Implementation

This comprehensive notebook covers the main biological and physical components of the VECTRI malaria model, combining **theory**, **equations**, **examples**, and **Python code** for each concept.

---

## 📋 Outline

1. [Introduction and Setup](#1.-Introduction-and-Setup)
2. [Larval Cycle](#2.-Larval-Cycle)
3. [Larval Mortality](#3.-Larval-Mortality)
4. [Gonotrophic Cycle](#4.-Gonotrophic-Cycle)
5. [Sporogonic Cycle](#5.-Sporogonic-Cycle)
6. [Vector Survival](#6.-Vector-Survival)
7. [Indoor Temperatures](#7.-Indoor-Temperatures)
8. [Host Community and Biting](#8.-Host-Community-and-Biting)
9. [Immunity](#9.-Immunity)
10. [Surface Hydrology](#10.-Surface-Hydrology)
11. [Complete Simulation Example](#11.-Complete-Simulation-Example)
12. [Exercises](#12.-Exercises)
13. [Summary](#13.-Summary)


---

## 1. Introduction and Setup

### 1.1 Required Libraries

First, let's import all the necessary libraries for this notebook.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set plot style for better visualization
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("Libraries imported successfully!")


### 1.2 Overview

VECTRI integrates multiple biological and physical processes:

- **Temperature-driven development** (larval, gonotrophic, sporogonic)
- **Mortality mechanisms** (crowding, flushing, lethal temperatures, temperature-dependent adult survival)
- **Host community structure** and **biting dynamics**
- **Immunity** that accumulates and decays with exposure
- **Rainfall-driven hydrology** determining breeding habitat availability


---

## 2. Larval Cycle

### 2.1 Concept

The larval cycle describes the development of mosquito larvae from hatching to adult emergence. VECTRI represents this as a **fractional life-cycle** from 0 to 1 and advances larvae along this axis each day.

Development is assumed to depend primarily on **water temperature** via a degree-day relationship:

- Warmer water → faster development
- No development below a minimum temperature
- Death above a maximum (lethal) temperature

### 2.2 Equations

The **larval development rate** $R_L$ (fraction of full development per day) is:

$$R_L = \frac{T_{wat} - T_{L,min}}{K_L}$$

where:
- $T_{wat}$ is the water temperature in breeding pools (°C)
- $T_{L,min}$ is the minimum temperature for larval development (°C)
- $K_L$ is the number of **degree-days** required to complete the larval stage

### 2.3 Parameters


In [ ]:
# Larval development parameters
T_L_min = 16.0   # [°C] Minimum water temperature for larval development
K_L = 90.9       # [degree-days] Degree-days needed to complete larval stage (Jepson)
T_L_max = 37.0   # [°C] Lethal upper water temperature (no larvae survive)

print(f"T_L_min = {T_L_min}°C")
print(f"K_L = {K_L} degree-days")
print(f"T_L_max = {T_L_max}°C")


### 2.4 Python Implementation


In [ ]:
def calculate_larval_development(T_wat, T_L_min=16.0, K_L=90.9, T_L_max=37.0):
    """
    Calculate larval development rate and period.
    
    Parameters:
    -----------
    T_wat : float
        Water temperature in breeding pools (°C)
    T_L_min : float
        Minimum temperature for larval development (°C)
    K_L : float
        Degree-days required to complete larval stage
    T_L_max : float
        Lethal upper temperature (°C)
    
    Returns:
    --------
    R_L : float
        Development rate (fraction per day)
    larval_period : float
        Days to complete larval development
    """
    if T_wat <= T_L_min or T_wat >= T_L_max:
        R_L = 0.0
        larval_period = math.inf
    else:
        R_L = (T_wat - T_L_min) / K_L
        larval_period = 1.0 / R_L
    
    return R_L, larval_period


### 2.5 Example Calculation

Let:
- $T_{wat} = 26°C$
- $T_{L,min} = 16°C$
- Case 1: $K_L = 90.9$ degree-days (Jepson)
- Case 2: $K_L = 200$ degree-days (Bayoh & Lindsay)


In [ ]:
# Example: Calculate larval development at 26°C
T_wat = 26.0

# Case 1: Jepson parameterization
R_L_case1, period_case1 = calculate_larval_development(T_wat, K_L=90.9)
print(f"Case 1 (K_L=90.9): R_L = {R_L_case1:.3f}, Period = {period_case1:.1f} days")

# Case 2: Bayoh & Lindsay parameterization
R_L_case2, period_case2 = calculate_larval_development(T_wat, K_L=200.0)
print(f"Case 2 (K_L=200): R_L = {R_L_case2:.3f}, Period = {period_case2:.1f} days")


### 2.6 Visualization: Temperature Sensitivity


In [ ]:
# Temperature range for analysis
temps = np.arange(10, 41, 1)
R_L_values = []
period_values = []

for T in temps:
    R_L, period = calculate_larval_development(T)
    R_L_values.append(R_L)
    period_values.append(period if period < 100 else np.nan)

# Plot development rate vs temperature
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(temps, R_L_values, 'b-', linewidth=2)
axes[0].axvline(x=16, color='r', linestyle='--', label='T_L_min = 16°C')
axes[0].axvline(x=37, color='r', linestyle='--', label='T_L_max = 37°C')
axes[0].set_xlabel('Water Temperature (°C)')
axes[0].set_ylabel('Development Rate R_L (fraction/day)')
axes[0].set_title('Larval Development Rate vs Temperature')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(temps, period_values, 'g-', linewidth=2)
axes[1].axvline(x=16, color='r', linestyle='--', label='T_L_min = 16°C')
axes[1].axvline(x=37, color='r', linestyle='--', label='T_L_max = 37°C')
axes[1].set_xlabel('Water Temperature (°C)')
axes[1].set_ylabel('Larval Period (days)')
axes[1].set_title('Larval Development Period vs Temperature')
axes[1].set_ylim(0, 50)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---

## 3. Larval Mortality

### 3.1 Concept

Larval survival depends on:
1. A **base survival rate** in "good" conditions
2. **Crowding** (resource limitation as biomass approaches pond carrying capacity)
3. **Flushing** by heavy rainfall
4. A **lethal temperature cutoff** at very high water temperatures

### 3.2 Equations

**Flushing Factor:**

$$K_{flush} = L_f + (1 - L_f) \left[ (1 - K_{flush,\infty}) e^{-R_d / \tau_{flush}} + K_{flush,\infty} \right]$$

**Total Larval Survival:**

$$P_{L,surv} = \left(1 - \frac{M_L}{w \cdot M_{L,max}}\right) K_{flush} \cdot P_{L,surv,0}$$

### 3.3 Parameters


In [ ]:
# Larval mortality parameters
P_L_surv0 = 0.825    # Base daily larval survival (no crowding, no flushing)
M_L_max = 300.0      # [mg m^-2] Larval biomass capacity (carrying capacity)
tau_flush = 50.0     # [mm/day] Rainfall scale for flushing
K_flush_inf = 0.4    # Survival of early larvae under very heavy rain (asymptote)

print(f"P_L_surv0 = {P_L_surv0}")
print(f"M_L_max = {M_L_max} mg/m²")
print(f"tau_flush = {tau_flush} mm/day")
print(f"K_flush_inf = {K_flush_inf}")


In [ ]:
def calculate_flushing_factor(L_f, R_d, tau_flush=50.0, K_flush_inf=0.4):
    """
    Calculate flushing factor based on larval stage and rainfall.
    """
    inner = (1.0 - K_flush_inf) * math.exp(-R_d / tau_flush) + K_flush_inf
    K_flush = L_f + (1.0 - L_f) * inner
    return K_flush

def calculate_larval_survival(M_L, w, R_d, L_f, P_L_surv0=0.825, M_L_max=300.0, 
                               tau_flush=50.0, K_flush_inf=0.4):
    """
    Calculate total larval survival probability.
    """
    if w <= 0.0:
        return 0.0, 0.0, 1.0
    
    # Crowding term
    crowd_term = 1.0 - M_L / (w * M_L_max)
    crowd_term = max(0.0, min(1.0, crowd_term))
    
    # Crowding-only survival
    P_L_surv_crowd = crowd_term * P_L_surv0
    
    # Flushing factor
    K_flush = calculate_flushing_factor(L_f, R_d, tau_flush, K_flush_inf)
    
    # Total survival
    P_L_surv = P_L_surv_crowd * K_flush
    P_L_surv = max(0.0, min(1.0, P_L_surv))
    
    return P_L_surv, P_L_surv_crowd, K_flush


In [ ]:
# Visualization: Flushing factor vs rainfall for different larval stages
rainfall = np.linspace(0, 100, 100)
stages = [0.0, 0.25, 0.5, 0.75, 1.0]

plt.figure(figsize=(10, 5))

for L_f in stages:
    K_flush_values = [calculate_flushing_factor(L_f, R) for R in rainfall]
    plt.plot(rainfall, K_flush_values, label=f'L_f = {L_f}', linewidth=2)

plt.xlabel('Daily Rainfall (mm/day)')
plt.ylabel('Flushing Factor K_flush')
plt.title('Flushing Factor vs Rainfall for Different Larval Stages')
plt.legend(title='Larval Stage')
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()


---

## 4. Gonotrophic Cycle

### 4.1 Concept

The gonotrophic cycle is the time from a mosquito's **blood meal** to **egg laying** (oviposition).

### 4.2 Equations

$$R_{gono} = \frac{T_{eff} - T_{gono,min}}{K_{gono}}$$

$$P_{gono} = \frac{1}{R_{gono}}$$


In [ ]:
# Gonotrophic cycle parameters
T_gono_min = 7.7    # [°C] Minimum temperature for egg development
K_gono = 37.1       # [degree-days] Degree-days needed to complete gonotrophic cycle

def calculate_gonotrophic_cycle(T_eff, T_gono_min=7.7, K_gono=37.1):
    """Calculate gonotrophic development rate and period."""
    if T_eff <= T_gono_min:
        R_gono = 0.0
        gono_period = math.inf
    else:
        R_gono = (T_eff - T_gono_min) / K_gono
        gono_period = 1.0 / R_gono
    return R_gono, gono_period

# Example at different temperatures
temperatures = [20, 25, 28]
for T in temperatures:
    R_gono, period = calculate_gonotrophic_cycle(T)
    print(f"T_eff = {T}°C: R_gono = {R_gono:.3f}, Period = {period:.1f} days")


---

## 5. Sporogonic Cycle

### 5.1 Concept

The sporogonic cycle describes the development of the malaria parasite inside the mosquito. The **extrinsic incubation period** (EIP) depends strongly on temperature.

### 5.2 Equations

$$R_{sporo} = \frac{T_{eff} - T_{sporo,min}}{K_{sporo}}$$

$$EIP = \frac{1}{R_{sporo}}$$


In [ ]:
# Sporogonic cycle parameters
T_sporo_min = 16.0  # [°C] Minimum temperature for parasite development
K_sporo = 111.0     # [degree-days] Degree-days for sporogonic cycle (EIP)
P_hv = 0.2          # P(mosquito infected | bite on infectious host)

def calculate_sporogonic_cycle(T_eff, T_sporo_min=16.0, K_sporo=111.0):
    """Calculate sporogonic development rate and EIP."""
    if T_eff <= T_sporo_min:
        R_sporo = 0.0
        EIP = math.inf
    else:
        R_sporo = (T_eff - T_sporo_min) / K_sporo
        EIP = 1.0 / R_sporo
    return R_sporo, EIP

# Example: EIP at different temperatures
temperatures = [18, 20, 25, 30]
for T in temperatures:
    R_sporo, EIP = calculate_sporogonic_cycle(T)
    print(f"T_eff = {T}°C: R_sporo = {R_sporo:.4f}, EIP = {EIP:.1f} days")


---

## 6. Vector Survival

### 6.1 Concept

Adult mosquito daily survival probability depends on temperature using the **Martens II** formulation.

### 6.2 Equations

$$P_{V,surv} = \exp\left(- \frac{1}{K_{0} + K_{1} T_{eff} + K_{2} T_{eff}^2}\right)$$

$$Lifespan \approx \frac{1}{1 - P_{V,surv}}$$


In [ ]:
# Vector survival parameters (Martens II)
K_mar2_0 = -4.4
K_mar2_1 = 1.31
K_mar2_2 = -0.03

def calculate_vector_survival(T_eff, K0=-4.4, K1=1.31, K2=-0.03):
    """Calculate adult vector daily survival probability and lifespan."""
    den = K0 + K1 * T_eff + K2 * (T_eff ** 2)
    
    if den <= 0.0:
        P_V_surv = 0.0
        lifespan = 0.0
    else:
        P_V_surv = math.exp(-1.0 / den)
        P_V_surv = max(0.0, min(1.0, P_V_surv))
        
        if P_V_surv >= 0.999:
            lifespan = math.inf
        elif P_V_surv <= 0.0:
            lifespan = 0.0
        else:
            lifespan = 1.0 / (1.0 - P_V_surv)
    
    return P_V_surv, lifespan

# Example at different temperatures
temperatures = [15, 20, 25, 30, 35]
for T in temperatures:
    P_surv, lifespan = calculate_vector_survival(T)
    print(f"T_eff = {T}°C: P_V_surv = {P_surv:.3f}, Lifespan = {lifespan:.1f} days")


---

## 7. Indoor Temperatures

### 7.1 Equations

$$T_{indoor} = T_0 + K \cdot T_{2m}$$

$$T_{eff} = \beta_{indoor} \cdot T_{indoor} + (1 - \beta_{indoor}) \cdot T_{2m}$$


In [ ]:
# Temperature parameters
T0_indoor = 10.33   # [°C] Intercept for indoor temperature
K_indoor = 0.58     # [-] Slope relating outdoor T to indoor T
beta_indoor = 0.5   # Fraction of time mosquitoes spend indoors

def calculate_temperatures(T2m, T0_indoor=10.33, K_indoor=0.58, beta_indoor=0.5, delta_Tw=1.5):
    """Calculate indoor, effective, and water temperatures."""
    T_indoor = T0_indoor + K_indoor * T2m
    T_eff = beta_indoor * T_indoor + (1.0 - beta_indoor) * T2m
    T_wat = T2m + delta_Tw
    return T_indoor, T_eff, T_wat

# Example
outdoor_temps = [15, 20, 25, 30, 35]
print("Outdoor | Indoor | Effective | Water")
print("-" * 45)
for T2m in outdoor_temps:
    T_indoor, T_eff, T_wat = calculate_temperatures(T2m)
    print(f"  {T2m}°C  |  {T_indoor:.1f}°C |   {T_eff:.1f}°C   | {T_wat:.1f}°C")


---

## 8. Host Community and Biting

### 8.1 Equations

**Human Biting Rate:**
$$\overline{hbr} = \left(1 - e^{-H / \tau_{zoo}}\right) \frac{V_b}{H}$$

**Daily EIR:**
$$EIR_d = \overline{hbr} \times CSPR$$

**Vector → Host Infection Probability:**
$$P_{v \to h} = 1 - e^{-EIR_d \cdot P_{vh}}$$


In [ ]:
# Host community parameters
tau_zoo = 50.0      # [people] Scale for zoophily/anthropophily
P_vh = 0.3          # P(host infected | infectious bite)

def calculate_biting_rate(V_biting, H, tau_zoo=50.0):
    """Calculate mean human biting rate."""
    if H <= 0.0:
        return 0.0
    phi = 1.0 - math.exp(-H / tau_zoo)
    hbr = phi * V_biting / H
    return hbr

def calculate_transmission(hbr, CSPR, P_vh=0.3):
    """Calculate daily EIR and vector-to-host infection probability."""
    EIR_d = hbr * CSPR
    P_v2h = 1.0 - math.exp(-EIR_d * P_vh)
    return EIR_d, P_v2h

# Example
V_biting = 50
H = 200
CSPR = 0.10

hbr = calculate_biting_rate(V_biting, H)
EIR_d, P_v2h = calculate_transmission(hbr, CSPR)

print(f"Human biting rate: {hbr:.3f} bites/person/day")
print(f"Daily EIR: {EIR_d:.4f} infectious bites/person/day")
print(f"Annual EIR: {EIR_d * 365:.1f} infectious bites/person/year")
print(f"Daily infection probability: {P_v2h:.4f} ({P_v2h*100:.2f}%)")


---

## 9. Immunity

### 9.1 Equations

$$\frac{dI}{dt} = \alpha \cdot EIR - \frac{I}{\tau}$$


In [ ]:
def update_immunity(I, EIR_annual, alpha=0.01, tau_years=3.0, dt=1.0):
    """Update immunity level based on exposure."""
    tau_days = tau_years * 365
    EIR_daily = EIR_annual / 365
    
    dI = (alpha * EIR_daily - I / tau_days) * dt
    I_new = I + dI
    I_new = max(0.0, min(1.0, I_new))
    
    return I_new

# Simulate immunity buildup and decay
days = 365 * 10  # 10 years
I = np.zeros(days)
EIR_pattern = np.zeros(days)

# High EIR for first 5 years, then zero
EIR_pattern[:365*5] = 100

for d in range(1, days):
    I[d] = update_immunity(I[d-1], EIR_pattern[d])

years = np.arange(days) / 365

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(years, EIR_pattern, 'b-', linewidth=2)
axes[0].set_ylabel('Annual EIR')
axes[0].set_title('Exposure Pattern')
axes[0].grid(True, alpha=0.3)

axes[1].plot(years, I, 'r-', linewidth=2)
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Immunity Level')
axes[1].set_title('Immunity Dynamics')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---

## 10. Surface Hydrology

### 10.1 Equations

$$\frac{dw_{pond}}{dt} = K_w \left[ P (w_{max} - w_{pond}) - w_{pond}(E + I) \right]$$


In [ ]:
# Hydrology parameters
w_max = 0.04        # Max fractional pond coverage (4% of grid cell)
E = 5.0             # [mm/day] Evaporation
I_infilt = 245.0    # [mm/day] Infiltration
K_w = 0.001         # Geometry/scale factor

def update_pond_fraction(w_prev, rain, w_max=0.04, E=5.0, I_infilt=245.0, K_w=0.001):
    """Update pond fraction based on water balance."""
    inflow = rain * (w_max - w_prev)
    outflow = w_prev * (E + I_infilt)
    
    dw = K_w * (inflow - outflow)
    w_new = w_prev + dw
    w_new = max(0.0, min(w_max, w_new))
    
    return w_new, inflow, outflow

# Simulate pond dynamics for 60 days with different rainfall scenarios
days = 60

scenarios = {
    'Constant (8 mm/day)': np.full(days, 8.0),
    'Episodic (30 mm/week)': np.where(np.arange(days) % 7 == 0, 30.0, 0.0),
    'Dry (1 mm/day)': np.full(days, 1.0)
}

plt.figure(figsize=(12, 5))

for name, rain in scenarios.items():
    w = np.zeros(days)
    w[0] = 0.01
    
    for d in range(1, days):
        w[d], _, _ = update_pond_fraction(w[d-1], rain[d])
    
    plt.plot(range(days), w * 100, label=name, linewidth=2)

plt.axhline(y=4, color='r', linestyle='--', label='w_max = 4%')
plt.xlabel('Day')
plt.ylabel('Pond Coverage (%)')
plt.title('Pond Dynamics Under Different Rainfall Scenarios')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---

## 11. Complete Simulation Example

Now let's put everything together in a complete simulation.


In [ ]:
# Create climate time series: 180 days
dates = pd.date_range("2025-01-01", periods=180, freq="D")
day_of_year = dates.dayofyear.values

# Synthetic air temperature [°C]: seasonal cycle around 24°C
T2m = 24.0 + 3.0 * np.sin(2 * np.pi * (day_of_year / 365.0))

# Synthetic rainfall [mm/day]: random Gamma distribution
rng = np.random.default_rng(42)
rain = rng.gamma(shape=2.0, scale=3.0, size=len(dates))

climate_df = pd.DataFrame({"T2m": T2m, "rain": rain}, index=dates)
print("Climate data created:")
print(climate_df.head())


In [ ]:
# Set location parameters
M_L = 3.0           # [mg m^-2] Larval biomass
L_f = 0.25          # Larval fractional stage
delta_Tw = 1.5      # Water temperature offset
H = 200.0           # Humans in the cell
H_inf_frac = 0.10   # Fraction infectious
V_biting = 50.0     # Biting mosquitoes
CSPR = 0.10         # Infectious fraction
beta_indoor = 0.5   # Fraction time indoors

# Run day-by-day simulation
n = len(climate_df)
w = np.zeros(n)
Twat = np.zeros(n)
T_eff_arr = np.zeros(n)
larval_period = np.zeros(n)
P_L_surv_arr = np.zeros(n)
gono_period_arr = np.zeros(n)
EIP_arr = np.zeros(n)
P_V_surv_arr = np.zeros(n)
lifespan_arr = np.zeros(n)
EIR_d_arr = np.zeros(n)

for i, (date, row) in enumerate(climate_df.iterrows()):
    T2 = row["T2m"]
    Rd = row["rain"]
    
    # Hydrology
    w_prev = w[i-1] if i > 0 else 0.0
    w[i], _, _ = update_pond_fraction(w_prev, Rd)
    
    # Temperatures
    T_indoor, T_eff, T_wat = calculate_temperatures(T2, beta_indoor=beta_indoor, delta_Tw=delta_Tw)
    Twat[i] = T_wat
    T_eff_arr[i] = T_eff
    
    # Larval development
    R_L, period = calculate_larval_development(T_wat)
    larval_period[i] = period if period < 100 else np.nan
    
    # Larval survival
    P_L_surv, _, _ = calculate_larval_survival(M_L, w[i], Rd, L_f)
    P_L_surv_arr[i] = P_L_surv
    
    # Gonotrophic cycle
    _, gono_p = calculate_gonotrophic_cycle(T_eff)
    gono_period_arr[i] = gono_p if gono_p < 20 else np.nan
    
    # Sporogonic cycle
    _, EIP = calculate_sporogonic_cycle(T_eff)
    EIP_arr[i] = EIP if EIP < 60 else np.nan
    
    # Vector survival
    P_V_surv, lifespan = calculate_vector_survival(T_eff)
    P_V_surv_arr[i] = P_V_surv
    lifespan_arr[i] = lifespan if lifespan < 50 else np.nan
    
    # Transmission
    hbr = calculate_biting_rate(V_biting, H)
    EIR_d, _ = calculate_transmission(hbr, CSPR)
    EIR_d_arr[i] = EIR_d

print("Simulation complete!")


In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(4, 2, figsize=(14, 12))

# Temperature dynamics
axes[0, 0].plot(dates, T2m, label="T2m (air)")
axes[0, 0].plot(dates, T_eff_arr, label="T_eff")
axes[0, 0].plot(dates, Twat, label="T_wat", linestyle="--")
axes[0, 0].set_ylabel("Temperature (°C)")
axes[0, 0].set_title("Temperature Dynamics")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Rainfall and ponds
ax1 = axes[0, 1]
ax2 = ax1.twinx()
ax1.bar(dates, rain, alpha=0.4, label="Rain")
ax2.plot(dates, w * 100, 'r-', label="Pond %", linewidth=2)
ax1.set_ylabel("Rain (mm/day)")
ax2.set_ylabel("Pond Coverage (%)")
ax1.set_title("Rainfall and Breeding Habitat")
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

# Larval survival
axes[1, 0].plot(dates, P_L_surv_arr)
axes[1, 0].set_ylabel("Survival Probability")
axes[1, 0].set_title("Larval Daily Survival")
axes[1, 0].set_ylim(0, 1)
axes[1, 0].grid(True, alpha=0.3)

# Development periods
axes[1, 1].plot(dates, larval_period, label="Larval")
axes[1, 1].plot(dates, gono_period_arr, label="Gonotrophic")
axes[1, 1].plot(dates, EIP_arr, label="EIP")
axes[1, 1].set_ylabel("Days")
axes[1, 1].set_title("Development Periods")
axes[1, 1].legend()
axes[1, 1].set_ylim(0, 30)
axes[1, 1].grid(True, alpha=0.3)

# Vector survival and lifespan
axes[2, 0].plot(dates, P_V_surv_arr)
axes[2, 0].set_ylabel("Daily Survival")
axes[2, 0].set_title("Adult Vector Survival")
axes[2, 0].set_ylim(0, 1)
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(dates, lifespan_arr)
axes[2, 1].set_ylabel("Days")
axes[2, 1].set_title("Expected Mosquito Lifespan")
axes[2, 1].grid(True, alpha=0.3)

# EIR
axes[3, 0].plot(dates, EIR_d_arr)
axes[3, 0].set_ylabel("Infectious bites/person/day")
axes[3, 0].set_title("Daily EIR")
axes[3, 0].grid(True, alpha=0.3)

# Summary statistics
stats_text = f"""
Summary Statistics:
------------------
Mean T_eff: {np.mean(T_eff_arr):.1f}°C
Mean Pond: {np.mean(w)*100:.2f}%
Mean Larval Survival: {np.mean(P_L_surv_arr):.3f}
Mean Vector Survival: {np.mean(P_V_surv_arr):.3f}
Mean EIP: {np.nanmean(EIP_arr):.1f} days
Mean Lifespan: {np.nanmean(lifespan_arr):.1f} days
"""
axes[3, 1].text(0.1, 0.5, stats_text, transform=axes[3, 1].transAxes, fontsize=12,
                verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[3, 1].axis('off')
axes[3, 1].set_title("Summary Statistics")

plt.tight_layout()
plt.show()


---

## 12. Exercises

### Exercise 1: Temperature Sensitivity Analysis

**Task**: Compare larval development with Jepson (K_L=90.9) vs Bayoh & Lindsay (K_L=200) parameterizations.


In [ ]:
# Exercise 1: Temperature Sensitivity Analysis
# Your code here - compare the two parameterizations

temps = np.arange(16, 38, 1)

# Calculate for both parameterizations
period_jepson = []
period_bayoh = []

for T in temps:
    _, p1 = calculate_larval_development(T, K_L=90.9)
    _, p2 = calculate_larval_development(T, K_L=200.0)
    period_jepson.append(p1 if p1 < 100 else np.nan)
    period_bayoh.append(p2 if p2 < 100 else np.nan)

plt.figure(figsize=(10, 5))
plt.plot(temps, period_jepson, 'b-', label='Jepson (K_L=90.9)', linewidth=2)
plt.plot(temps, period_bayoh, 'r-', label='Bayoh & Lindsay (K_L=200)', linewidth=2)
plt.xlabel('Water Temperature (°C)')
plt.ylabel('Larval Development Period (days)')
plt.title('Comparison of Larval Development Parameterizations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Exercise 2: EIP vs Mosquito Lifespan

**Task**: Determine when parasites can complete development before mosquitoes die.


In [ ]:
# Exercise 2: EIP vs Mosquito Lifespan
temps = np.arange(16, 36, 0.5)
EIP_vals = []
lifespan_vals = []

for T in temps:
    _, EIP = calculate_sporogonic_cycle(T)
    _, lifespan = calculate_vector_survival(T)
    EIP_vals.append(EIP if EIP < 60 else np.nan)
    lifespan_vals.append(lifespan if lifespan < 50 else np.nan)

plt.figure(figsize=(10, 5))
plt.plot(temps, EIP_vals, 'r-', label='EIP (parasite development)', linewidth=2)
plt.plot(temps, lifespan_vals, 'b-', label='Mosquito lifespan', linewidth=2)
plt.fill_between(temps, 0, 50, where=[e < l if not (np.isnan(e) or np.isnan(l)) else False 
                                       for e, l in zip(EIP_vals, lifespan_vals)],
                 alpha=0.3, color='green', label='Transmission possible')
plt.xlabel('Temperature (°C)')
plt.ylabel('Days')
plt.title('EIP vs Mosquito Lifespan: Transmission Window')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 50)
plt.tight_layout()
plt.show()


### Exercise 3: Rainfall Scenarios

**Task**: Understand how different rainfall patterns affect breeding habitat and larval survival.


In [ ]:
# Exercise 3: Rainfall Scenarios
days = 60
M_L = 3.0
L_f = 0.25

scenarios = {
    'Constant (5 mm/day)': np.full(days, 5.0),
    'Episodic (25 mm/5 days)': np.where(np.arange(days) % 5 == 0, 25.0, 0.0),
    'Dry (0.5 mm/day)': np.full(days, 0.5)
}

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

for idx, (name, rain) in enumerate(scenarios.items()):
    w = np.zeros(days)
    survival = np.zeros(days)
    w[0] = 0.01
    
    for d in range(1, days):
        w[d], _, _ = update_pond_fraction(w[d-1], rain[d])
        survival[d], _, _ = calculate_larval_survival(M_L, w[d], rain[d], L_f)
    
    ax1 = axes[idx]
    ax2 = ax1.twinx()
    
    ax1.bar(range(days), rain, alpha=0.3, color='blue', label='Rainfall')
    ax1.plot(range(days), w * 100, 'g-', label='Pond %', linewidth=2)
    ax2.plot(range(days), survival, 'r-', label='Larval Survival', linewidth=2)
    
    ax1.set_ylabel('Rain (mm) / Pond (%)')
    ax2.set_ylabel('Survival Probability')
    ax1.set_title(f'{name} - Mean Survival: {np.mean(survival[1:]):.3f}')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax2.set_ylim(0, 1)

axes[-1].set_xlabel('Day')
plt.tight_layout()
plt.show()


---

## 13. Summary

This comprehensive notebook covered VECTRI's main components:

| Component | Key Equation | Temperature Dependence |
|-----------|-------------|------------------------|
| Larval Development | $R_L = (T_{wat} - T_{L,min}) / K_L$ | Degree-days in water |
| Larval Survival | Crowding × Flushing × Base | Lethal threshold |
| Gonotrophic Cycle | $R_{gono} = (T_{eff} - T_{gono,min}) / K_{gono}$ | Effective temperature |
| Sporogonic Cycle | $R_{sporo} = (T_{eff} - T_{sporo,min}) / K_{sporo}$ | Effective temperature |
| Vector Survival | Martens II formula | Bell-shaped curve |
| Hydrology | Water balance ODE | Evaporation |

**Key Insights**:
- Temperature controls development rates and survival
- Rainfall creates breeding habitat but can flush larvae
- The EIP must be shorter than mosquito lifespan for transmission
- Human population density affects biting rates
- Immunity modulates clinical outcomes

---

## 🔗 Additional Resources

- [VECTRI Online Documentation](https://users.ictp.it/~tompkins/vectri/documentation/)
- Tompkins & Ermert (2013) - A regional-scale, high resolution dynamical malaria model
- Bayoh & Lindsay (2003) - Effect of temperature on the development of the aquatic stages of Anopheles gambiae
- Martens et al. (1995) - Potential impact of global climate change on malaria risk
